# Exploring a fairness audit runA guided tour of what the pipeline produces and how to interrogate it. Everythinghere reads **committed artifacts** — no model is called, so nothing in thisnotebook costs quota or can change the numbers in the report.If you have no run yet:```bashfrs pipeline --config configs/offline_demo.yaml```

In [ ]:
import json
from pathlib import Path

# `frs` is installed by `uv sync`, so no sys.path juggling is needed.
# Launch this notebook with:  uv run jupyter lab
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

import warnings
warnings.simplefilter("ignore")

import pandas as pd
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)

from frs.config import Config
from frs.pipeline import load_run, latest_run

cfg = Config.load(ROOT / "configs" / "offline_demo.yaml", apply_env=False)
run_id = latest_run(cfg)
print("latest run:", run_id)
state = load_run(cfg, run_id)
print("arms available:", sorted(state.frames_by_arm))

## 1. First question: can the scorer be trusted?Fairness numbers from a scorer that does not discriminate on job-relevant contentare worthless — a constant function is perfectly unbiased and perfectly useless.So this comes before any bias metric.

In [ ]:
from frs.fairness import variance_decomposition, self_consistency, parse_qualityframe = state.frames_by_arm["baseline"]print("score records:", len(frame))print()print("Phase-2 discrimination gate:")print(json.dumps(state.sanity, indent=1)[:700])

In [ ]:
reliability = variance_decomposition(frame)print(json.dumps({k: v for k, v in reliability.items() if k != "interpretation"}, indent=1))print()print(reliability["interpretation"])

In [ ]:
consistency = self_consistency(frame)print(consistency["interpretation"])print()quality = parse_quality(frame)print("parse ok rate       :", quality["parse_ok_rate"])print("evidence grounding  :", quality.get("mean_evidence_support_rate"))for warning in quality["warnings"]:    print("WARNING:", warning)

### Reading the noise floor`cue_effect_to_noise_ratio` is the number that decides whether a cue effect is afinding. Below 1.0, cue-driven variation is smaller than the model's ownrepeat-to-repeat variation, and any threshold failure is the scorer disagreeingwith itself rather than reacting to the cue.

## 2. What each cue does to the scoreEvery value is centred on the candidate's **own** baseline, so resume strength isdifferenced out and name cues and non-name cues are directly comparable — which isthe comparison this project exists to make.

In [ ]:
from frs.fairness import cue_effect_tableeffects = cue_effect_table(frame)by_family = (    effects.groupby("cue_families")    .agg(n_levels=("cue_ids", "nunique"),         mean_delta=("mean_delta", "mean"),         worst_delta=("mean_delta", "min"),         best_delta=("mean_delta", "max"))    .sort_values("worst_delta"))by_family.round(3)

In [ ]:
# The ten cue levels that cost a candidate the most.effects.sort_values("mean_delta").head(10)[    ["cue_families", "cue_ids", "n_candidates", "mean_delta", "se_delta"]].round(3)

## 3. Individual fairness: would *this* candidate's score change?Group means are an average over people. A rejected candidate is not asking aboutthe average — they are asking whether their own score moved.

In [ ]:
from frs.fairness import counterfactual_invarianceanalysis = counterfactual_invariance(frame, tolerance=2.0, n_boot=500)print(analysis["interpretation"])print()for key in ("mean_max_swing", "p90_max_swing", "worst_max_swing",            "decision_flip_rate", "decision_flip_rate_from_noise",            "decision_flip_rate_above_noise"):    print(f"{key:<32} {analysis[key]}")

In [ ]:
# The candidates whose scores moved most under identity cues alone.pd.DataFrame(analysis["worst_candidates"])[    ["job_id", "base_resume_id", "n_variants", "max_swing", "decision_flipped"]].round(3)

## 4. Group parity, done for a blocked designEach resume is a statistical *block*: the same document appears under every cuelevel. Differencing within block removes the between-resume variance, which is byfar the largest component — and which a pooled t-test would leave in thedenominator.

In [ ]:
from frs.fairness import group_parityfor attribute in ("gender", "race_ethnicity"):    result = group_parity(frame, attribute, reference_level=None,                          n_boot=500, n_permutations=1000)    if "error" in result:        print(f"{attribute}: {result['error']}")        continue    print(f"=== {attribute} ===")    print("centred means:", result["centred_means"])    print("largest pairwise gap:", result["max_pairwise_gap"],          "| omnibus permutation p:", round(result["omnibus_permutation"]["p_value"], 4))    for comparison in result["comparisons"]:        print(f"  {comparison['level']:<14} vs {comparison['reference']:<8} "              f"gap {comparison['gap']:>7} "              f"CI [{comparison['gap_ci_low']:>6}, {comparison['gap_ci_high']:>6}] "              f"perm p {comparison['permutation_p']:<8} "              f"MDE {comparison['minimum_detectable_effect']}")    print()

### Why the MDE column mattersWhere the minimum detectable effect exceeds the practical threshold, a null resultis a statement about sample size, not about fairness. *Absence of evidence is notevidence of absence* — and the conflict analyser flags exactly this case as`power_masquerade`.

## 5. Allocational fairness: who actually gets shortlistedA score gap is distributional. What a candidate experiences is whether they madethe top *k*.

In [ ]:
from frs.fairness import allocational_analysisfor k in (1, 3, 5):    result = allocational_analysis(frame, "gender", k=k)    if "error" in result:        print(result["error"]); continue    print(f"top-{k}: rates {result['selection_rate']} "          f"| impact ratio {result['impact_ratio']} "          f"| four-fifths {result['four_fifths_rule_satisfied']}")    if result.get("tie_warning"):        print("   ", result["tie_warning"])

## 6. Does anonymisation remove the signal?The project's central claim, measured rather than asserted: apply each redactionlevel, then ask the cue detector how much demographic signal it can still recover.

In [ ]:
from frs.anonymize import compare_levelsrows = []for resume in state.resumes:    for entry in compare_levels(resume):        rows.append(entry)leakage = pd.DataFrame(rows).groupby("level", sort=False).mean(numeric_only=True).round(2)leakage.loc[["none", "identity", "identity_plus_proxy", "aggressive"]]

Anything still inferable in the `identity` row is a cue that name-level blindscreening — the industry-standard intervention — does not remove.

## 7. Auditing the model's *explanations*A screener can give two identical resumes identical scores and still describe onecandidate as "talented" and the other as "hardworking". The recruiter reads therationale, so that asymmetry reaches the decision even when the number does not.

In [ ]:
from frs.fairness import lexicon_rates, distinctive_termsrates = lexicon_rates(frame, "gender")if "error" in rates:    print(rates["error"])else:    display(pd.DataFrame(rates["by_level"]).T[        ["n_rationales", "agentic_minus_communal", "ability_minus_effort",         "hedge_minus_booster", "protected_mention_rate"]    ].round(3))terms = distinctive_terms(frame, "gender", top_n=6)if "distinctive_terms" in terms:    for level, entries in terms["distinctive_terms"].items():        print(level, "->", ", ".join(f"{e['term']}({e['z']})" for e in entries))

## 8. Metric conflictsSeveral fairness criteria are provably incompatible. A report whose metrics allagree is either measuring one thing repeatedly or has not looked hard enough.

In [ ]:
from frs.fairness import run_audit, audit_summary, headline_findingsreport = run_audit(frame, cfg, run_id=run_id)summary = audit_summary(report)print("metrics:", summary["n_metrics"],      "| failed:", summary["n_failed"],      "| significant after FDR:", summary["n_significant_after_fdr"])print()pd.DataFrame(headline_findings(report))[["metric", "value", "unit", "verdict"]]

In [ ]:
for conflict in report.conflicts:    print(f"[{conflict['severity'].upper()}] {conflict['type']}")    print("   ", conflict["explanation"][:320])    print("    -> ", conflict["recommendation"][:220])    print()

## 9. Mitigation: what helped, and what it costThe two easiest ways to pass a fairness audit are to score everyone identically andto decline to decide. Both are caught by the utility guards.

In [ ]:
path = state.root / "mitigation" / "utility_costs.csv"if path.exists():    display(pd.read_csv(path).round(3))else:    print("no mitigation arms in this run; enable mitigation.strategies in the config")

## 10. Explanation cardsAggregate metrics describe a population; a recruiter decides about a person. Cardsare ordered most cue-sensitive first, because those are the decisions a human mostneeds to look at.

In [ ]:
cards_path = state.root / "report" / "explanation_cards.json"if cards_path.exists():    cards = json.loads(cards_path.read_text())    print(f"{len(cards)} cards; "          f"{sum(1 for c in cards if c['cue_sensitivity']['crosses_decision_boundary'])} "          "decision-unstable")    from frs.report import render_card_markdown    from IPython.display import Markdown    display(Markdown(render_card_markdown(cards[0])))else:    print("no cards in this run")

## 11. Where to go next| question | try || --- | --- || does the audit suite itself work? | `frs validate-auditor --provider mock` || what if only names are perturbed? | `frs pipeline --config configs/names_only.yaml` || what does a real corpus look like? | `frs pipeline --config configs/kaggle.yaml` || how does a real model behave? | add `GEMINI_API_KEY` to `.env`, then `frs pipeline` || explore interactively | `frs dashboard` |Full method and its limitations: `docs/METHODOLOGY.md`.